# MODELLING

## Apply Logistic Regression, Random Forest and HistGradBoosted for each df - encode → scale → fit → predict → evaluate

In [1]:
# import libraries
import pandas as pd
import numpy as np

# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Train/test split
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
)

# Plotting (for ROC curves, feature importance, etc.)
import matplotlib.pyplot as plt

In [2]:
# read in dfs

model_df_binary = pd.read_parquet("../data/model_df_binary.parquet")
model_df_multiclass = pd.read_parquet("../data/model_df_multiclass.parquet")



# test train split time

In [3]:
print(model_df_binary.columns.tolist())
print(model_df_multiclass.columns.tolist())

['crime_id', 'month', 'reported_by', 'longitude', 'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type', 'last_outcome_category', 'outcome_binary', 'employ_score_rate', 'income_score_rate', 'living_environment_score', 'barriers_score']
['crime_id', 'month', 'reported_by', 'longitude', 'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type', 'last_outcome_category', 'outcome_binary', 'employ_score_rate', 'income_score_rate', 'living_environment_score', 'barriers_score', 'outcome_multiclass_grouped']


In [4]:
print(model_df_binary.columns.tolist())

['crime_id', 'month', 'reported_by', 'longitude', 'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type', 'last_outcome_category', 'outcome_binary', 'employ_score_rate', 'income_score_rate', 'living_environment_score', 'barriers_score']


In [5]:
#create the combined stratification column (NEW — add this first)
model_df_binary['strat_col'] = (
    model_df_binary['crime_type'].astype(str) + '_' + model_df_binary['outcome_binary'].astype(str)
)

In [6]:
#Build X/y for the binary dataframe. This will leave X with just crime_type and income_score_rate 

X = model_df_binary.drop(columns=[
    'crime_id',
    'month',
    'reported_by',
    'longitude',
    'latitude',
    'location',
    'lsoa_code',
    'lsoa_name',
    'last_outcome_category',
    'outcome_binary',
    'employ_score_rate',
    'living_environment_score',
    'barriers_score',
    'strat_col'
])
y = model_df_binary['outcome_binary']

In [7]:
# Build X_multi/y_multi for the multiclass dataframe. This should have X_multi ending up with just crime_type and income_score_rate

X_multi = model_df_multiclass.drop(columns=[
    'crime_id',
    'month',
    'reported_by',
    'longitude',
    'latitude',
    'location',
    'lsoa_code',
    'lsoa_name',
    'last_outcome_category',
    'employ_score_rate',
    'living_environment_score',
    'barriers_score',
    'outcome_multiclass_grouped'
])
y_multi = model_df_multiclass['outcome_multiclass_grouped']




In [8]:
# Note strat_col stays out of X, but remains in model_df_binary so can reference it for stratification.

In [9]:
# TEST/TRAIN SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=model_df_binary['strat_col']
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

outcome_binary
0    0.923074
1    0.076926
Name: proportion, dtype: float64
outcome_binary
0    0.923097
1    0.076903
Name: proportion, dtype: float64


BINARY SPLIT DONE Both results above come out close to the overall 7.69% resolved / 92.31% unresolved split hopefully

In [10]:
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

#check counts. count for each outcome category, smallest first. As long as the smallest category has at least 2 rows, the split will succeed 

y_multi.value_counts().sort_values()

outcome_multiclass_grouped
Other                                                      23
Formal action is not in the public interest               499
Further investigation is not in the public interest       917
Offender given a caution                                 1936
Action to be taken by another organisation               3667
Further action is not in the public interest             3996
Local resolution                                         6465
Investigation complete; no suspect identified          100941
Unable to prosecute suspect                            109113
Name: count, dtype: int64

NOTE: there are  only 23 rows in Other, an 80/20 split approx, 18 in train and 5 in test = a small sample for the model to learn that category from, and an even smaller one to evaluate it on — so Other might ends up with unstable or unreliable precision/recall numbers later, due to having so few examples . A limitation to note if Other's metrics look erratic.

In [11]:
# TEST/TRAIN SPLIT FOR MULTICLASS DF

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi,
    test_size=0.2,
    random_state=42,
    stratify=y_multi
)

In [12]:
print(y_train_multi.value_counts(normalize=True))
print(y_test_multi.value_counts(normalize=True))

outcome_multiclass_grouped
Unable to prosecute suspect                            0.479497
Investigation complete; no suspect identified          0.443583
Local resolution                                       0.028411
Further action is not in the public interest           0.017562
Action to be taken by another organisation             0.016117
Offender given a caution                               0.008509
Further investigation is not in the public interest    0.004032
Formal action is not in the public interest            0.002192
Other                                                  0.000099
Name: proportion, dtype: float64
outcome_multiclass_grouped
Unable to prosecute suspect                            0.479500
Investigation complete; no suspect identified          0.443597
Local resolution                                       0.028410
Further action is not in the public interest           0.017556
Action to be taken by another organisation             0.016106
Offender given a 

So at this point you have, fully built and checked:
BinaryMulticlassFeaturesX (crime_type, income_score_rate)X_multi (same two columns)Targety (outcome_binary)y_multi (outcome_multiclass_grouped)SplitX_train, X_test, y_train, y_testX_train_multi, X_test_multi, y_train_multi, y_test_multiStratified oncrime_type + outcome_binary combinedoutcome_multiclass_grouped aloneVerifiedproportions match to ~0.01%proportions match to ~0.01%

Multiclass split done - The above shows a clean match every category's proportion in train and test lines up almost exactly (down to the 4th decimal place), including the tiny Other category at ~0.01%. The stratification worked correctly across all nine outcome categories


THE LIST OF NEW VARIABLES - because I always forget!

**Binary dataframe (`model_df_binary`)**

| Variable | Contents |
|---|---|
| `X` | Features: `crime_type`, `income_score_rate` |
| `y` | Target: `outcome_binary` |
| `X_train` | Training features (80%) |
| `X_test` | Test features (20%) |
| `y_train` | Training target (80%) |
| `y_test` | Test target (20%) |

Stratified on: `model_df_binary['strat_col']` (`crime_type` + `outcome_binary` combined)

**Multiclass dataframe (`model_df_multiclass`)**

| Variable | Contents |
|---|---|
| `X_multi` | Features: `crime_type`, `income_score_rate` |
| `y_multi` | Target: `outcome_multiclass_grouped` |
| `X_train_multi` | Training features (80%) |
| `X_test_multi` | Test features (20%) |
| `y_train_multi` | Training target (80%) |
| `y_test_multi` | Test target (20%) |

Stratified on: `y_multi` alone (`outcome_multiclass_grouped`)

# One hot encoding - crime_type

In [13]:
# need to encode it on X_train first, then apply the same encoding to X_test.

# one-hot encode crime_type on training data
X_train_encoded = pd.get_dummies(X_train, columns=['crime_type'], drop_first=True)

# apply the same encoding to test data, aligning columns to match X_train_encoded
X_test_encoded = pd.get_dummies(X_test, columns=['crime_type'], drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

# NOTE: The reindex(columns=X_train_encoded.columns, fill_value=0) line is the important safety step — it forces X_test_encoded to have exactly the same dummy columns as X_train_encoded, in the same order, filling in 0 for any category that didn't happen to appear in the test set. Without this, if test has a crime type train doesn't (or vice versa), your two encoded dataframes could end up with different numbers of columns, which would break the model when you try to predict.

In [14]:
# check the shapes match up
X_train_encoded.shape[1] == X_test_encoded.shape[1]

True

# scale income_score_rate for use in logistic regression 

In [15]:
from sklearn.preprocessing import StandardScaler

# instantiate the scaler
scaler = StandardScaler()

# fit on training data only, then transform it
X_train_encoded['income_score_rate'] = scaler.fit_transform(X_train_encoded[['income_score_rate']])

# use the already-fitted scaler to transform test data (no fitting here)
X_test_encoded['income_score_rate'] = scaler.transform(X_test_encoded[['income_score_rate']])

In [16]:
print(X_train_encoded['income_score_rate'].mean(), X_train_encoded['income_score_rate'].std())
print(X_test_encoded['income_score_rate'].mean(), X_test_encoded['income_score_rate'].std())

-8.368234542201722e-17 1.0000027465698773
0.0032043355275436607 0.9988118316778521


## Fit the logistic regression model (binary)

In [17]:
from sklearn.linear_model import LogisticRegression

# instantiate the model, using class_weight='balanced' to handle the 7.69%/92.31% imbalance
log_reg = LogisticRegression(class_weight='balanced', random_state=42)

# fit on the encoded, scaled training data
log_reg.fit(X_train_encoded, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default

In [18]:
# PREDICT ON THE TEST SET

# predicted class labels (0 = unresolved, 1 = resolved)
y_pred = log_reg.predict(X_test_encoded)

# predicted probabilities (needed for ROC-AUC and precision-recall curves)
y_pred_proba = log_reg.predict_proba(X_test_encoded)[:, 1]

In [19]:
#Evaluation

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Classification report
print(classification_report(y_test, y_pred, target_names=['Unresolved', 'Resolved']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\nROC-AUC: {roc_auc:.4f}")

              precision    recall  f1-score   support

  Unresolved       0.95      0.85      0.90     42012
    Resolved       0.22      0.50      0.30      3500

    accuracy                           0.82     45512
   macro avg       0.58      0.68      0.60     45512
weighted avg       0.90      0.82      0.85     45512


Confusion Matrix:
[[35639  6373]
 [ 1739  1761]]

ROC-AUC: 0.7649


In [20]:
from sklearn.metrics import average_precision_score
pr_auc = average_precision_score(y_test, y_pred_proba)
print(f"PR-AUC: {pr_auc:.4f}")

PR-AUC: 0.4112


# Random Forest Baseline - Binary Classification

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train_encoded, y_train)


#Same class_weight='balanced' and random_state=42 as LR, for a fair comparison

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.

NOTES: No n_estimators or max_depth specified — these have been left  at sklearn defaults (n_estimators=100, no max depth) deliberately, since this is meant to be a baseline, not a tuned model. To tune later, that would be a separate step after baselines are all in so nt done any hyperparameter tuning here

In [22]:
# predict/evaluate step

y_pred_rf = rf.predict(X_test_encoded)
y_pred_proba_rf = rf.predict_proba(X_test_encoded)[:, 1]

print(classification_report(y_test, y_pred_rf, target_names=['Unresolved', 'Resolved']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba_rf):.4f}")

              precision    recall  f1-score   support

  Unresolved       0.96      0.75      0.84     42012
    Resolved       0.18      0.63      0.27      3500

    accuracy                           0.74     45512
   macro avg       0.57      0.69      0.56     45512
weighted avg       0.90      0.74      0.80     45512


Confusion Matrix:
[[31681 10331]
 [ 1297  2203]]

ROC-AUC: 0.7766
PR-AUC: 0.4224


# HISTGRADIENTBOOSTING

HistGradientBoosting is a tree-based ensemble method that builds decision trees sequentially, where each new tree focuses on correcting the errors of the trees built before it, using histogram-binned features for speed on large datasets


unlike RF, which builds many independent trees and averages them (reducing variance), HGB builds trees that learn from each other's mistakes (reducing bias) — so where RF improves robustness, HGB is built to chase down patterns the earlier trees missed, which makes it a meaningfully different third comparison point rather than just "another tree model" alongside RF

In [23]:
# execture HGB fit cell
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
hgb.fit(X_train_encoded, y_train)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form `{class_label: weight}`.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas `n_samples / (n_classes * np.bincount(y))`.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if `sample_weight` is specified... versionadded:: 1.2",'balanced'
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [24]:
y_pred_hgb = hgb.predict(X_test_encoded)
y_pred_proba_hgb = hgb.predict_proba(X_test_encoded)[:, 1]

print(classification_report(y_test, y_pred_hgb, target_names=['Unresolved', 'Resolved']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_hgb))
print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba_hgb):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba_hgb):.4f}")

              precision    recall  f1-score   support

  Unresolved       0.95      0.87      0.91     42012
    Resolved       0.24      0.51      0.33      3500

    accuracy                           0.84     45512
   macro avg       0.60      0.69      0.62     45512
weighted avg       0.90      0.84      0.86     45512


Confusion Matrix:
[[36409  5603]
 [ 1717  1783]]

ROC-AUC: 0.7834
PR-AUC: 0.4305


# Ablation Testing 

I will probably do 3 ablation runs - all on HGB because that is probably the most useful of the 3 models

1. feature removal of crime_type
2. feature removal of income_deprivation
3. class weighting removal - this will refit HGB without class_weight='balanced' (i.e. default). This isolates how much of the recall-on-minority-class performance is coming from the class weighting itself versus the model's inherent ability to separate the classes.

In [26]:
# Ablation 1: drop income_score_rate, keep crime_type only 

# build the feature subset — crime_type columns only, no income_score_rate
income_cols = ['income_score_rate']
X_train_no_income = X_train_encoded.drop(columns=income_cols)
X_test_no_income = X_test_encoded.drop(columns=income_cols)

hgb_no_income = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)
hgb_no_income.fit(X_train_no_income, y_train)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form `{class_label: weight}`.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas `n_samples / (n_classes * np.bincount(y))`.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if `sample_weight` is specified... versionadded:: 1.2",'balanced'
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
